<a href="https://colab.research.google.com/github/Heba2627/NLP-Project/blob/main/NLP_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Importing Dataset via. Pandas

In [1]:
import pandas as pd
import numpy as np
import regex as re

# Login using e.g. `huggingface-cli login` to access this dataset
splits = {'train': 'train.parquet', 'validation': 'validation.parquet', 'test': 'test.parquet'}

In [2]:
#Fetching train, validation and test dataframes.
train_df = pd.read_parquet("hf://datasets/coastalcph/tydi_xor_rc/" + splits["train"])
val_df   = pd.read_parquet("hf://datasets/coastalcph/tydi_xor_rc/" + splits["validation"])
test_df  = pd.read_parquet("hf://datasets/coastalcph/tydi_xor_rc/" + splits["test"])

#I NEED TO REPLACE ALL THE "df" REFERENCES WITH "train_df" AT SOME POINT.
df = train_df
print(df.columns)
answerable_df = df[df["answerable"] == True]
unanswerable_df = df[df["answerable"] == False]

print(len(answerable_df))
print(len(unanswerable_df))
print("% of dataset that is answerable:", len(answerable_df)/len(df)*100, "%")
print("% of dataset that is unanswerable:", len(unanswerable_df)/len(df)*100, "%")

Index(['question', 'context', 'lang', 'answerable', 'answer_start', 'answer',
       'answer_inlang'],
      dtype='str')
13962
1381
% of dataset that is answerable: 90.99915270807534 %
% of dataset that is unanswerable: 9.000847291924655 %


In [3]:
#Check for letters in all languages + Mark for arabic
word_pattern = re.compile(r"[\p{L}\p{M}]+", flags=re.UNICODE)

def n_tokens(text):
    return len(word_pattern.findall(str(text)))
def median_iqr(s):
    q1, median, q3 = np.percentile(s, [25, 50, 75])
    return pd.Series({"median": median, "Q1": q1, "Q3": q3, "IQR": q3 - q1})

df["question_len"] = df["question"].apply(n_tokens)
df["context_len"] = df["context"].apply(n_tokens)

#Approach idea: build a mini DataFrame to obtain a table like described in the project pdf.
length_stats = pd.DataFrame({"question_length": median_iqr(df["question_len"]), "context_length": median_iqr(df["context_len"])})
length_stats

,question_length,context_length
median,5.0,84.0
Q1,4.0,56.0
Q3,7.0,122.0
IQR,3.0,66.0


In [4]:
#missing values
missing = df.isna().sum().to_frame("n_missing")
missing["pct_missing"] = (100 * missing["n_missing"]/len(df))
missing

#duplicates
dup_filter = df.duplicated(subset=["question", "context"], keep=False)
dup_amount = dup_filter.sum()
dup_groups = df[dup_filter].groupby(["question","context"]).ngroups
redudant_rows = df.duplicated(subset=["question", "context"], keep="first").sum()
print(dup_groups)
print(dup_amount)
print(redudant_rows)

17
34
17


# Tokenisation

In [5]:
!pip install deep_translator

In [6]:
from transformers import AutoTokenizer
from deep_translator import GoogleTranslator
from collections import Counter, defaultdict

tok = AutoTokenizer.from_pretrained("xlm-roberta-base")

#We need to root out symbols so check for letter
has_letter = re.compile(r"\p{L}", flags=re.UNICODE)

#NOTE: We will probably need a special case for japanese since its the only
#language that doesnt have whitespace or punctuation!
def tokenize_words_xlmr(text):
    pieces = tok.tokenize(str(text))
    words, current = [], ""
    for p in pieces:
        if p.startswith("▁"):
            if current != "":
                words.append(current)
            current = p[1:]
        else:
            current += p
    if current != "":
        words.append(current)
    return [w.lower() for w in words if w != "" and has_letter.search(w)]

def translate(token, lang):
    try:
      return GoogleTranslator(source=lang, target="en").translate(token)
    except:
      return None

counters = defaultdict(Counter)
for lang, question in zip(df["lang"], df["question"]):
    counters[lang].update(tokenize_words_xlmr(question))

rows = [
    {"lang": lang, "token": token, "count": count}
    for lang, counter in counters.items()
    for token, count in counter.most_common(5)
]
top5_df = pd.DataFrame(rows)
top5_df["In English"] = [translate(token, lang) for token, lang in zip(top5_df["token"], top5_df["lang"])]
top5_df = top5_df[["lang", "token", "In English", "count"]]
top5_df

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

c:\Users\alska\AppData\Local\Programs\Python\Python313\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\alska\.cache\huggingface\hub\models--xlm-roberta-base. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

,lang,token,In English,count
0,bn,হয়,is,467
1,bn,প্রথম,the first,458
2,bn,কী,what,439
3,bn,কে,who,415
4,bn,কত,how much,392
5,ja,pygが最初に発表した曲は何,What was the first song released by pyg?,3
6,ja,オーロラ号の漂流時、オーロラ号に乗組員は何人いた?,How many people were on board the Aurora when ...,3
7,ja,カンザスシティが設立されたのはいつ,When was Kansas City founded?,3
8,ja,クロイツ群が初めて発見されたのはいつ,When was the Kreutz group first discovered?,3
9,ja,ライド,ride,2


In [7]:
#Task 3
def verify_answerable(df):
    checked = 0
    failures = []
    for i, row in enumerate(df):
        if row["answerable"]: 
            checked+= 1
            context=row["context"]
            answer=row["answer"]
            start=row["answer_start"]

            expected=context[start:start+len(answer)]
            if expected!=answer:
                failures.append({
                    "index": i,
                    "answer_start": start,
                    "answer": answer,
                    "substring_found": expected,
                    "context": context})
    return checked,failures

#The pdf said to apply it to both splits
#we have to use ar,ko and te only, and they exist only in train and validation.
checked_train,failures_train=verify_answerable(train_df.to_dict("records"))
print(f"Checked {checked_train} answerable rows in the training set, found {len(failures_train)} failures")

checked_val,failures_val=verify_answerable(val_df.to_dict("records"))
print(f"Checked {checked_val} answerable rows in the validation set, found {len(failures_val)} failures")


Checked 13962 answerable rows in the training set, found 0 failures
Checked 2313 answerable rows in the validation set, found 0 failures


# i) Majority Baseline

In [8]:
majority_label = df["answerable"].value_counts().idxmax()
print("Majority class form training split: ", majority_label)

def majority_baseline(question=None, Context=None):
    return majority_label

Majority class form training split:  True


In [9]:
val_pred = [majority_baseline(q, c) for q, c in zip(val_df["question"], val_df["context"])]
acc_score = (pd.Series(val_pred).values == val_df["answerable"].values).mean()
print("Accuracy of Majority Baseline on Validation split: ", acc_score*100, "%")

Accuracy of Majority Baseline on Validation split:  76.81833277980738 %


## ii) Determinstic rule-based classifier baseline 

- Raw tokenization overlapping between questions and contexts would never work(different languages)
- Whatever approach I take will struggle with the semintically equivalent words like created and invented, as we can't have semantic understanding without a model(unless I bring a synonym dictionary)
- What I am going to do is to basically translate the question using deep-translator's google translator, tokenize the question and the context, removing stop words, put into lower case, and possibly lemmatize, and then I will depend on lexical overlap between the tokens but not the pure thing(because of the semantically equivalent words problem), so I will try to map some question words to their expected answers using regex, and then I will use the overlap score between the question and the context to determine the answerability.
- What it can exploit is the samples that survives the translation, named entities, numbers, dates, and question-word typing. These will survive because proper nouns and numerals are often unchanged, and question-word categories are a small closed set you can pattern-match directly.
- What it can't exploit is the mistranslations, negations and basically anyhting requires semantic understanding.

In [20]:
import string

tokenizer= AutoTokenizer.from_pretrained("xlm-roberta-base")

STOPWORDS= {
    "the", "a", "an", "is", "are", "was", "were", "of", "in", "on", "at",
    "to", "for", "and", "or", "that", "this", "these", "those", "by", "with",
    "from", "as", "it", "its"
}

QUESTION_TYPE_PATTERNS= {
    "person":re.compile(r"\bwho\b", re.I),
    "date":re.compile(r"\bwhen\b", re.I),
    "location":re.compile(r"\bwhere\b", re.I),
    "number":re.compile(r"\bhow many\b|\bhow much\b", re.I),
}

ENTITY_PATTERNS= {
    "person":re.compile(r"\b[A-Z][a-z]+(?:\s[A-Z][a-z]+)+\b"),
    "date":re.compile(r"\b\d{1,2}(st|nd|rd|th)?\s\w+\s\d{4}\b|\b\d{4}\b|\b\w+\s\d{1,2},\s\d{4}\b"),
    "location":re.compile(r"\b[A-Z][a-z]+\b"),
    "number":re.compile(r"\b\d+([.,]\d+)?\b"),
}


def translate_to_english(text,lang):
    try:
        return GoogleTranslator(source=lang, target="en").translate(text)
    except Exception:
        return text 


def clean_tokens(text):
    tokens= tokenizer.tokenize(text.lower())
    tokens= [token for token in tokens if token and token not in string.punctuation]
    tokens= [token for token in tokens if token not in STOPWORDS]
    return set(tokens)


def lexical_overlap_score(question_en, context):
    q_tokens= clean_tokens(question_en)
    c_tokens= clean_tokens(context)
    if not q_tokens:
        return 0.0
    return len(q_tokens&c_tokens)/len(q_tokens)


def detect_question_type(question_en):
    for qtype, pattern in QUESTION_TYPE_PATTERNS.items():
        if pattern.search(question_en):
            return qtype
    return "other"


def context_has_type_match(context, qtype):
    if qtype == "other":
        return False
    pattern= ENTITY_PATTERNS.get(qtype)
    return bool(pattern.search(context)) if pattern else False


def predict_answerable(question, context, source_lang="auto"):
    question_en= translate_to_english(question, lang=source_lang)
    overlap= lexical_overlap_score(question_en, context)
    qtype= detect_question_type(question_en)
    type_match= context_has_type_match(context, qtype) 
    prediction= (overlap>=0.1) or type_match
    return prediction

In [21]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
validation_df= val_df[val_df["lang"].isin(["ar", "ko", "te"])]

def evaluate_classifier(val_df):
    y_true, y_pred= [], []

    for row in val_df.to_dict("records"):
        pred = predict_answerable(
            row["question"], row["context"],
            source_lang=row["lang"],
        )
        y_true.append(bool(row["answerable"]))
        y_pred.append(pred)



    metrics = {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred),
        "recall": recall_score(y_true, y_pred),
        "f1": f1_score(y_true, y_pred),
        "confusion_matrix": confusion_matrix(y_true, y_pred).tolist(),
    }
    return metrics



metrics= evaluate_classifier(validation_df)
print(f"Accuracy:{metrics['accuracy']:.3f}")
print(f"Precision:{metrics['precision']:.3f}")
print(f"Recall:{metrics['recall']:.3f}")
print(f"F1:{metrics['f1']:.3f}")
print(f"Confusion matrix [[TN, FP], [FN, TP]]: {metrics['confusion_matrix']}")

Token indices sequence length is longer than the specified maximum sequence length for this model (611 > 512). Running this sequence through the model will result in indexing errors


Accuracy:0.694
Precision:0.894
Recall:0.731
F1:0.804
Confusion matrix [[TN, FP], [FN, TP]]: [[78, 86], [267, 724]]
